# NER Herbal — BiLSTM + Naive Bayes Split-Safe (Fixed)

Notebook ini sudah diperiksa ulang untuk dataset `bio_tagging_herbal_clean_BIO.csv`.

Perbaikan utama:
- Bug `apply_threshold` yang belum terdefinisi sudah diperbaiki.
- BiLSTM dan Naive Bayes memakai split train/validation/test yang sama.
- Balancing hanya dilakukan pada data train.
- Validation dan test tetap clean agar evaluasi tidak bocor.
- Artifact untuk web disimpan: model BiLSTM, model Naive Bayes, dan mapping.


In [1]:
# =========================================================
# 1. INSTALL & IMPORT
# =========================================================

!pip -q install seqeval

import os
import re
import random
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.pipeline import Pipeline
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
TensorFlow: 2.20.0


In [2]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2


In [3]:
# =========================================================
# 2. LOAD CLEAN DATASET ONLY
# =========================================================

# Di Colab, upload file: bio_tagging_herbal_clean_BIO.csv
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

if "balanced" in str(csv_path).lower():
    raise ValueError(
        "File yang diupload tampak seperti dataset balanced. "
        "Untuk notebook split-safe ini, upload dataset clean asli: "
        "bio_tagging_herbal_clean_BIO.csv"
    )

df = pd.read_csv(csv_path, encoding="utf-8-sig")

required_cols = ["SentenceID", "Token", "Label"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}. Kolom tersedia: {list(df.columns)}")

df = df[required_cols].copy()
df["SentenceID"] = pd.to_numeric(df["SentenceID"], errors="raise").astype(int)
df["Token"] = df["Token"].astype(str)
df["Label"] = df["Label"].astype(str)

print("File:", csv_path)
print("Shape:", df.shape)
print("Jumlah SentenceID:", df["SentenceID"].nunique())

print("\nPreview:")
display(df.head())

print("\nLabel distribution:")
display(df["Label"].value_counts().to_frame("count"))

o_ratio = (df["Label"] == "O").mean()
print("\nO ratio:", round(o_ratio, 4))

if o_ratio < 0.75:
    raise ValueError(
        "Rasio O terlalu rendah untuk dataset clean asli. "
        "Kemungkinan kamu mengupload dataset balanced."
    )


Saving Dataset_NER_BIO_Tagging_Final_Terkoreksi.csv to Dataset_NER_BIO_Tagging_Final_Terkoreksi.csv
File: Dataset_NER_BIO_Tagging_Final_Terkoreksi.csv
Shape: (10015, 3)
Jumlah SentenceID: 463

Preview:


,SentenceID,Token,Label
0,1,Diabetes,B-DISEASE
1,1,mellitus,I-DISEASE
2,1,adalah,O
3,1,penyakit,O
4,1,metabolisme,O



Label distribution:


,count
Label,
O,8240
B-HERB,284
I-HERB,276
B-COMPOUND,223
B-TREATMENT,221
B-EFFECT,151
I-EFFECT,111
B-METHOD,102
I-TREATMENT,80



O ratio: 0.8228


In [4]:
# =========================================================
# 3. BIO VALIDATION
# =========================================================

VALID_ENTITY_TYPES = {
    "HERB", "DISEASE", "COMPOUND", "BODY_PART",
    "EFFECT", "TREATMENT", "METHOD", "POPULATION"
}

def validate_bio_dataframe(df):
    invalid_labels = []
    bio_violations = []

    for idx, row in df.iterrows():
        label = row["Label"]
        if label == "O":
            continue
        if not (label.startswith("B-") or label.startswith("I-")):
            invalid_labels.append((idx, row["SentenceID"], row["Token"], label))
            continue
        entity_type = label[2:]
        if entity_type not in VALID_ENTITY_TYPES:
            invalid_labels.append((idx, row["SentenceID"], row["Token"], label))

    for sid, group in df.groupby("SentenceID", sort=False):
        prev_label = "O"
        for idx, row in group.iterrows():
            label = row["Label"]
            if label.startswith("I-"):
                entity_type = label[2:]
                valid_prev = {f"B-{entity_type}", f"I-{entity_type}"}
                if prev_label not in valid_prev:
                    bio_violations.append({
                        "index": idx,
                        "SentenceID": sid,
                        "Token": row["Token"],
                        "Label": label,
                        "Previous_Label": prev_label
                    })
            prev_label = label

    return invalid_labels, bio_violations

invalid_labels, bio_violations = validate_bio_dataframe(df)

print("Invalid labels:", len(invalid_labels))
print("BIO violations:", len(bio_violations))

if len(invalid_labels) == 0 and len(bio_violations) == 0:
    print("Dataset valid BIO.")
else:
    if invalid_labels:
        display(pd.DataFrame(invalid_labels, columns=["index", "SentenceID", "Token", "Label"]).head(20))
    if bio_violations:
        display(pd.DataFrame(bio_violations).head(20))


Invalid labels: 0
BIO violations: 0
Dataset valid BIO.


In [5]:
# =========================================================
# 4. GROUP SENTENCES & SPLIT ORIGINAL CLEAN DATA
# =========================================================

LOWER_TOKEN = True

def normalize_token(token):
    token = str(token)
    return token.lower() if LOWER_TOKEN else token

sentences = []
labels = []
sentence_ids = []

for sid, group in df.groupby("SentenceID", sort=False):
    tokens = [normalize_token(t) for t in group["Token"].tolist()]
    tags = group["Label"].tolist()
    sentences.append(tokens)
    labels.append(tags)
    sentence_ids.append(sid)

X_train, X_temp, y_train, y_temp, sid_train, sid_temp = train_test_split(
    sentences, labels, sentence_ids,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

X_val, X_test, y_val, y_test, sid_val, sid_test = train_test_split(
    X_temp, y_temp, sid_temp,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train clean sentences:", len(X_train))
print("Val clean sentences:", len(X_val))
print("Test clean sentences:", len(X_test))

print("Train ∩ Val :", len(set(sid_train) & set(sid_val)))
print("Train ∩ Test:", len(set(sid_train) & set(sid_test)))
print("Val ∩ Test  :", len(set(sid_val) & set(sid_test)))


Train clean sentences: 370
Val clean sentences: 46
Test clean sentences: 47
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0


In [6]:
# =========================================================
# 5. SPLIT DISTRIBUTION
# =========================================================

def count_tags(tag_sequences):
    c = Counter()
    for seq in tag_sequences:
        c.update(seq)
    return pd.Series(dict(c)).sort_values(ascending=False)

print("TRAIN clean label distribution:")
display(count_tags(y_train).to_frame("count"))

print("VALIDATION clean label distribution:")
display(count_tags(y_val).to_frame("count"))

print("TEST clean label distribution:")
display(count_tags(y_test).to_frame("count"))


TRAIN clean label distribution:


,count
O,6579
B-HERB,233
I-HERB,223
B-TREATMENT,172
B-COMPOUND,168
B-EFFECT,123
I-EFFECT,90
B-METHOD,82
I-TREATMENT,64
B-DISEASE,61


VALIDATION clean label distribution:


,count
O,787
B-TREATMENT,23
I-HERB,21
B-HERB,20
B-EFFECT,14
B-COMPOUND,12
I-TREATMENT,9
I-EFFECT,8
B-METHOD,8
B-DISEASE,7


TEST clean label distribution:


,count
O,874
B-COMPOUND,43
I-HERB,32
B-HERB,31
B-TREATMENT,26
B-EFFECT,14
I-EFFECT,13
B-METHOD,12
B-DISEASE,9
B-POPULATION,9


In [7]:
# =========================================================
# 6. MAKE BALANCED TRAIN DATA FROM TRAIN SPLIT ONLY
# =========================================================

ENTITY_DUP_FACTOR = {
    "BODY_PART": 8,
    "POPULATION": 6,
    "METHOD": 1,
    "TREATMENT": 3,
    "DISEASE": 3,
    "EFFECT": 5,
    "COMPOUND": 4,
    "HERB": 2,
}

WINDOW_SIZE = 8
KEEP_ORIGINAL_TRAIN_SENTENCES = True

def repair_bio_tags(tags):
    fixed = []
    prev = "O"
    for tag in tags:
        if tag.startswith("I-"):
            entity = tag[2:]
            if prev not in [f"B-{entity}", f"I-{entity}"]:
                tag = f"B-{entity}"
        fixed.append(tag)
        prev = tag
    return fixed

def get_entity_types(tags):
    return [tag[2:] for tag in tags if tag.startswith("B-")]

def sentence_duplication_factor(tags):
    factor = 1
    for entity in get_entity_types(tags):
        factor = max(factor, ENTITY_DUP_FACTOR.get(entity, 1))
    return factor

def make_entity_windows(tokens, tags, window_size=8):
    windows = []
    for i, tag in enumerate(tags):
        if not tag.startswith("B-"):
            continue
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)
        win_tokens = tokens[start:end]
        win_tags = repair_bio_tags(tags[start:end])
        if any(t != "O" for t in win_tags):
            windows.append((win_tokens, win_tags))
    return windows

def build_balanced_train(X_train, y_train):
    X_bal, y_bal = [], []
    for tokens, tags in zip(X_train, y_train):
        if KEEP_ORIGINAL_TRAIN_SENTENCES:
            factor = sentence_duplication_factor(tags)
            for _ in range(factor):
                X_bal.append(tokens)
                y_bal.append(tags)

        windows = make_entity_windows(tokens, tags, WINDOW_SIZE)
        for win_tokens, win_tags in windows:
            factor = sentence_duplication_factor(win_tags)
            for _ in range(factor):
                X_bal.append(win_tokens)
                y_bal.append(win_tags)
    return X_bal, y_bal

X_train_bal, y_train_bal = build_balanced_train(X_train, y_train)

print("Train clean sentences :", len(X_train))
print("Train balanced samples:", len(X_train_bal))

print("\nBalanced train label distribution:")
display(count_tags(y_train_bal).to_frame("count"))

clean_train_counts = count_tags(y_train)
bal_train_counts = count_tags(y_train_bal)
print("\nO ratio clean train:", round(clean_train_counts.get("O", 0) / clean_train_counts.sum(), 4))
print("O ratio balanced train:", round(bal_train_counts.get("O", 0) / bal_train_counts.sum(), 4))


Train clean sentences : 370
Train balanced samples: 5609

Balanced train label distribution:


,count
O,67915
B-HERB,3652
I-HERB,3465
B-COMPOUND,3339
B-TREATMENT,2688
B-EFFECT,2346
I-EFFECT,1617
B-BODY_PART,1552
B-POPULATION,1250
B-DISEASE,1191



O ratio clean train: 0.8226
O ratio balanced train: 0.7373


In [8]:
# =========================================================
# 7. BUILD VOCAB & TAG MAPPING
# =========================================================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_TAG = "<PAD>"

word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}

for sent in X_train_bal:
    for token in sent:
        if token not in word2idx:
            word2idx[token] = len(word2idx)

tag2idx = {PAD_TAG: 0}

for tag in sorted(df["Label"].unique().tolist()):
    if tag not in tag2idx:
        tag2idx[tag] = len(tag2idx)

idx2tag = {idx: tag for tag, idx in tag2idx.items()}

MAX_LEN = max(len(s) for s in sentences)

print("Vocab size:", len(word2idx))
print("Tag size:", len(tag2idx))
print("MAX_LEN:", MAX_LEN)
print(tag2idx)


Vocab size: 1680
Tag size: 18
MAX_LEN: 69
{'<PAD>': 0, 'B-BODY_PART': 1, 'B-COMPOUND': 2, 'B-DISEASE': 3, 'B-EFFECT': 4, 'B-HERB': 5, 'B-METHOD': 6, 'B-POPULATION': 7, 'B-TREATMENT': 8, 'I-BODY_PART': 9, 'I-COMPOUND': 10, 'I-DISEASE': 11, 'I-EFFECT': 12, 'I-HERB': 13, 'I-METHOD': 14, 'I-POPULATION': 15, 'I-TREATMENT': 16, 'O': 17}


In [9]:
# =========================================================
# 8. ENCODE & PAD
# =========================================================

def encode_pad_sequences(X, y, word2idx, tag2idx, max_len):
    X_ids = np.zeros((len(X), max_len), dtype=np.int32)
    y_ids = np.zeros((len(y), max_len), dtype=np.int32)

    for i, (sent, tags) in enumerate(zip(X, y)):
        length = min(len(sent), max_len)
        for j in range(length):
            X_ids[i, j] = word2idx.get(sent[j], word2idx[UNK_TOKEN])
            y_ids[i, j] = tag2idx.get(tags[j], tag2idx["O"])
    return X_ids, y_ids

X_train_ids, y_train_ids = encode_pad_sequences(X_train_bal, y_train_bal, word2idx, tag2idx, MAX_LEN)
X_val_ids, y_val_ids = encode_pad_sequences(X_val, y_val, word2idx, tag2idx, MAX_LEN)
X_test_ids, y_test_ids = encode_pad_sequences(X_test, y_test, word2idx, tag2idx, MAX_LEN)

print("X_train:", X_train_ids.shape)
print("X_val  :", X_val_ids.shape)
print("X_test :", X_test_ids.shape)


X_train: (5609, 69)
X_val  : (46, 69)
X_test : (47, 69)


In [10]:
# =========================================================
# 9. WEIGHTED LOSS
# =========================================================

def build_class_weight_vector(y_train_sequences, tag2idx):
    counts = Counter()
    for seq in y_train_sequences:
        counts.update(seq)

    total = sum(counts.values())
    n_classes_without_pad = len(tag2idx) - 1
    weights = np.ones(len(tag2idx), dtype=np.float32)

    for tag, idx in tag2idx.items():
        if tag == PAD_TAG:
            weights[idx] = 0.0

        elif tag == "O":
            weights[idx] = 0.40

        elif tag in ["B-METHOD", "I-METHOD"]:
            raw_weight = total / (n_classes_without_pad * max(counts.get(tag, 1), 1))
            weights[idx] = np.clip(raw_weight, 1.0, 2.5)

        elif tag in ["B-EFFECT", "I-EFFECT"]:
            raw_weight = total / (n_classes_without_pad * max(counts.get(tag, 1), 1))
            weights[idx] = np.clip(raw_weight, 2.0, 8.0)

        elif tag in ["B-COMPOUND", "I-COMPOUND"]:
            raw_weight = total / (n_classes_without_pad * max(counts.get(tag, 1), 1))
            weights[idx] = np.clip(raw_weight, 2.0, 7.0)

        elif tag in ["B-POPULATION", "I-POPULATION"]:
            raw_weight = total / (n_classes_without_pad * max(counts.get(tag, 1), 1))
            weights[idx] = np.clip(raw_weight, 2.0, 8.0)

        elif tag in ["B-TREATMENT", "I-TREATMENT"]:
            raw_weight = total / (n_classes_without_pad * max(counts.get(tag, 1), 1))
            weights[idx] = np.clip(raw_weight, 1.0, 5.0)

        else:
            raw_weight = total / (n_classes_without_pad * max(counts.get(tag, 1), 1))
            weights[idx] = np.clip(raw_weight, 1.0, 6.0)

    return weights


class_weight_vector = build_class_weight_vector(y_train_bal, tag2idx)

print("Class weights:")
for idx, w in enumerate(class_weight_vector):
    print(f"{idx2tag[idx]:15s} -> {w:.3f}")

class_weight_tensor = tf.constant(class_weight_vector, dtype=tf.float32)
pad_tag_id = tag2idx[PAD_TAG]


def weighted_sparse_categorical_crossentropy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)

    base_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    weights = tf.gather(class_weight_tensor, y_true)

    mask = tf.cast(tf.not_equal(y_true, pad_tag_id), tf.float32)

    weighted_loss = base_loss * weights * mask

    return tf.reduce_sum(weighted_loss) / tf.reduce_sum(mask)

Class weights:
<PAD>           -> 0.000
B-BODY_PART     -> 3.491
B-COMPOUND      -> 2.000
B-DISEASE       -> 4.550
B-EFFECT        -> 2.310
B-HERB          -> 1.484
B-METHOD        -> 2.500
B-POPULATION    -> 4.335
B-TREATMENT     -> 2.016
I-BODY_PART     -> 6.000
I-COMPOUND      -> 7.000
I-DISEASE       -> 6.000
I-EFFECT        -> 3.351
I-HERB          -> 1.564
I-METHOD        -> 2.500
I-POPULATION    -> 8.000
I-TREATMENT     -> 5.000
O               -> 0.400


In [11]:
# =========================================================
# 10. BUILD BILSTM MODEL
# =========================================================

VOCAB_SIZE = len(word2idx)
TAG_SIZE = len(tag2idx)

EMBED_DIM = 128
LSTM_UNITS = 128
DROPOUT_RATE = 0.30
LEARNING_RATE = 0.001

tf.keras.backend.clear_session()

inputs = tf.keras.layers.Input(shape=(MAX_LEN,), name="input_ids")

x = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    input_length=MAX_LEN,
    mask_zero=True,
    name="embedding"
)(inputs)

x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE)(x)

x = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(
        LSTM_UNITS,
        return_sequences=True,
        dropout=DROPOUT_RATE,
        recurrent_dropout=0.0
    ),
    name="bilstm"
)(x)

x = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Dense(64, activation="relu")
)(x)

outputs = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Dense(TAG_SIZE, activation="softmax")
)(x)

bilstm_model = tf.keras.Model(inputs=inputs, outputs=outputs)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=weighted_sparse_categorical_crossentropy,
    metrics=["accuracy"]
)

bilstm_model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 69)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 69, 128)   │    215,040 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 69, 128)   │          0 │ embedding[0][0]   │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 69)        │          0 │ input_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, 69, 256)   │    263,168 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 69, 64)    │     16,448 │ bilstm[0][0],     │
│ (TimeDistributed)   │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 69, 18)    │      1,170 │ time_distributed… │
│ (TimeDistributed)   │                   │            │ not_equal[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 495,826 (1.89 MB)

 Trainable params: 495,826 (1.89 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# =========================================================
# 11. TRAIN BILSTM
# =========================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )
]

history = bilstm_model.fit(
    X_train_ids,
    y_train_ids,
    validation_data=(X_val_ids, y_val_ids),
    batch_size=32,
    epochs=40,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 34ms/step - accuracy: 0.8514 - loss: 1.2404 - val_accuracy: 0.9688 - val_loss: 0.2271 - learning_rate: 0.0010
Epoch 2/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.9871 - loss: 0.0581 - val_accuracy: 0.9795 - val_loss: 0.1350 - learning_rate: 0.0010
Epoch 3/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.9952 - loss: 0.0205 - val_accuracy: 0.9774 - val_loss: 0.1347 - learning_rate: 0.0010
Epoch 4/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.9976 - loss: 0.0121 - val_accuracy: 0.9763 - val_loss: 0.1320 - learning_rate: 0.0010
Epoch 5/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.9982 - loss: 0.0088 - val_accuracy: 0.9784 - val_loss: 0.1724 - learning_rate: 0.0010
Epoch 6/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.9990 - loss: 0.0049 - val_accuracy: 0.9828 - val_loss: 0.1741 - learning_rate: 0.0010
Epoch 7/40
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.9996 - loss: 0

In [13]:
# =========================================================
# 12. BILSTM EVALUATION HELPERS
# =========================================================

def decode_bilstm_predictions(pred_ids, true_ids, idx2tag):
    y_true_text = []
    y_pred_text = []

    for true_seq, pred_seq in zip(true_ids, pred_ids):
        true_tags = []
        pred_tags = []

        for true_id, pred_id in zip(true_seq, pred_seq):
            true_tag = idx2tag[int(true_id)]

            if true_tag == PAD_TAG:
                continue

            pred_tag = idx2tag[int(pred_id)]

            if pred_tag == PAD_TAG:
                pred_tag = "O"

            true_tags.append(true_tag)
            pred_tags.append(pred_tag)

        y_true_text.append(true_tags)
        y_pred_text.append(pred_tags)

    return y_true_text, y_pred_text


def fix_bio_sequence(tags):
    fixed = []
    prev = "O"

    for tag in tags:
        if tag.startswith("I-"):
            entity = tag[2:]
            if prev not in [f"B-{entity}", f"I-{entity}"]:
                tag = f"B-{entity}"

        fixed.append(tag)
        prev = tag

    return fixed


def apply_threshold(pred_proba, threshold):
    pred_ids = np.argmax(pred_proba, axis=-1)
    pred_conf = np.max(pred_proba, axis=-1)

    o_id = tag2idx["O"]
    pred_ids_thresholded = pred_ids.copy()

    for i in range(pred_ids.shape[0]):
        for j in range(pred_ids.shape[1]):
            pred_tag = idx2tag[int(pred_ids[i, j])]

            if pred_tag == "O" or pred_tag == PAD_TAG:
                continue

            if pred_conf[i, j] < threshold:
                pred_ids_thresholded[i, j] = o_id

    return pred_ids_thresholded


CLASS_SPECIFIC_THRESHOLDS = {
    "B-EFFECT": 0.60,
    "I-EFFECT": 0.60,
    "B-TREATMENT": 0.52,
    "I-TREATMENT": 0.52,
    "B-METHOD": 0.35,
    "I-METHOD": 0.35,
    "B-POPULATION": 0.40,
    "I-POPULATION": 0.40,
}


def apply_class_specific_threshold(pred_proba, default_threshold=0.45):
    pred_ids = np.argmax(pred_proba, axis=-1)
    pred_conf = np.max(pred_proba, axis=-1)

    o_id = tag2idx["O"]
    pred_ids_thresholded = pred_ids.copy()

    for i in range(pred_ids.shape[0]):
        for j in range(pred_ids.shape[1]):
            pred_tag = idx2tag[int(pred_ids[i, j])]

            if pred_tag == "O" or pred_tag == PAD_TAG:
                continue

            threshold = CLASS_SPECIFIC_THRESHOLDS.get(pred_tag, default_threshold)

            if pred_conf[i, j] < threshold:
                pred_ids_thresholded[i, j] = o_id

    return pred_ids_thresholded


In [14]:
# =========================================================
# 13. THRESHOLD SEARCH ON CLEAN VALIDATION SET
# =========================================================

val_pred_proba = bilstm_model.predict(X_val_ids, verbose=0)
threshold_results = []

for th in np.arange(0.40, 0.76, 0.05):
    val_pred_ids_th = apply_threshold(val_pred_proba, th)

    y_true_val, y_pred_val = decode_bilstm_predictions(
        val_pred_ids_th,
        y_val_ids,
        idx2tag
    )

    y_pred_val = [fix_bio_sequence(seq) for seq in y_pred_val]

    threshold_results.append({
        "threshold": round(float(th), 2),
        "precision": precision_score(y_true_val, y_pred_val),
        "recall": recall_score(y_true_val, y_pred_val),
        "f1": f1_score(y_true_val, y_pred_val)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.sort_values("f1", ascending=False))

BEST_THRESHOLD = float(threshold_df.sort_values("f1", ascending=False).iloc[0]["threshold"])
print("Best threshold:", BEST_THRESHOLD)


,threshold,precision,recall,f1
6,0.70,0.911111,0.872340,0.891304
5,0.65,0.881720,0.872340,0.877005
7,0.75,0.898876,0.851064,0.874317
4,0.60,0.864583,0.882979,0.873684
3,0.55,0.848485,0.893617,0.870466
2,0.50,0.831683,0.893617,0.861538
0,0.40,0.831683,0.893617,0.861538
1,0.45,0.831683,0.893617,0.861538


Best threshold: 0.7


In [15]:
# =========================================================
# 14. FINAL TEST BILSTM ON CLEAN UNSEEN TEST SET
# =========================================================

USE_CLASS_SPECIFIC_THRESHOLD = True

test_pred_proba = bilstm_model.predict(X_test_ids, verbose=0)

if USE_CLASS_SPECIFIC_THRESHOLD:
    test_pred_ids = apply_class_specific_threshold(
        test_pred_proba,
        default_threshold=BEST_THRESHOLD
    )
else:
    test_pred_ids = apply_threshold(
        test_pred_proba,
        BEST_THRESHOLD
    )

y_true_bilstm, y_pred_bilstm = decode_bilstm_predictions(
    test_pred_ids,
    y_test_ids,
    idx2tag
)

y_pred_bilstm = [fix_bio_sequence(seq) for seq in y_pred_bilstm]

print("FINAL CLEAN TEST - BILSTM")
print("Best Threshold:", BEST_THRESHOLD)
print("Class-specific threshold:", USE_CLASS_SPECIFIC_THRESHOLD)
print("BiLSTM Precision:", precision_score(y_true_bilstm, y_pred_bilstm))
print("BiLSTM Recall   :", recall_score(y_true_bilstm, y_pred_bilstm))
print("BiLSTM F1       :", f1_score(y_true_bilstm, y_pred_bilstm))

print("\nClassification Report BiLSTM:")
print(classification_report(y_true_bilstm, y_pred_bilstm, digits=4))


FINAL CLEAN TEST - BILSTM
Best Threshold: 0.7
Class-specific threshold: True
BiLSTM Precision: 0.8333333333333334
BiLSTM Recall   : 0.8223684210526315
BiLSTM F1       : 0.8278145695364237

Classification Report BiLSTM:
              precision    recall  f1-score   support

   BODY_PART     1.0000    0.5000    0.6667         8
    COMPOUND     0.9474    0.8372    0.8889        43
     DISEASE     0.5000    0.3333    0.4000         9
      EFFECT     0.5238    0.7857    0.6286        14
        HERB     0.9677    0.9677    0.9677        31
      METHOD     0.5714    0.6667    0.6154        12
  POPULATION     0.9000    1.0000    0.9474         9
   TREATMENT     0.9231    0.9231    0.9231        26

   micro avg     0.8333    0.8224    0.8278       152
   macro avg     0.7917    0.7517    0.7547       152
weighted avg     0.8522    0.8224    0.8281       152



In [16]:
# =========================================================
# ERROR ANALYSIS BILSTM TERBARU
# =========================================================

error_rows = []

for sid, tokens, true_tags, pred_tags in zip(sid_test, X_test, y_test, y_pred_bilstm):
    for token, true_tag, pred_tag in zip(tokens, true_tags, pred_tags):
        if true_tag != pred_tag and (true_tag != "O" or pred_tag != "O"):
            error_rows.append({
                "SentenceID": sid,
                "Token": token,
                "True_Label": true_tag,
                "Pred_Label": pred_tag
            })

df_errors = pd.DataFrame(error_rows)

print("Total errors:", len(df_errors))

error_summary = (
    df_errors.groupby(["True_Label", "Pred_Label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(error_summary.head(30))

Total errors: 46


,True_Label,Pred_Label,count
18,O,B-EFFECT,7
6,B-DISEASE,O,4
5,B-COMPOUND,O,3
12,I-DISEASE,O,3
1,B-BODY_PART,O,2
3,B-COMPOUND,B-METHOD,2
0,B-BODY_PART,I-EFFECT,2
21,O,B-TREATMENT,2
16,O,B-COMPOUND,2
9,B-METHOD,O,2


# Naive Bayes Split-Safe

Naive Bayes dipakai sebagai model pembanding berbasis token features.

Skema:
```text
train = clean train split
balancing = hanya token train
test = clean unseen test split
```


In [17]:
# =========================================================
# 16. NAIVE BAYES FEATURE EXTRACTION
# =========================================================

def token_shape(token):
    token = str(token)

    if token.isdigit():
        return "digit"
    if token.isupper():
        return "upper"
    if token.istitle():
        return "title"
    if token.islower():
        return "lower"

    return "mixed"


def word2features(sent, i):
    word = str(sent[i])

    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word[-1:]": word[-1:],
        "word[-2:]": word[-2:],
        "word[-3:]": word[-3:],
        "word[:1]": word[:1],
        "word[:2]": word[:2],
        "word.isdigit": word.isdigit(),
        "word.shape": token_shape(word),
        "position": i,
        "relative_position": round(i / max(len(sent), 1), 2),
    }

    if i > 0:
        prev_word = str(sent[i - 1])
        features.update({
            "prev.word.lower": prev_word.lower(),
            "prev.word[-3:]": prev_word[-3:],
            "prev.word.shape": token_shape(prev_word),
        })
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        next_word = str(sent[i + 1])
        features.update({
            "next.word.lower": next_word.lower(),
            "next.word[-3:]": next_word[-3:],
            "next.word.shape": token_shape(next_word),
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def flatten_features_labels(X, y):
    features = []
    labels_flat = []

    for sent, tags in zip(X, y):
        sent_features = sent2features(sent)

        for feat, tag in zip(sent_features, tags):
            features.append(feat)
            labels_flat.append(tag)

    return features, labels_flat


X_train_nb_features, y_train_nb_flat = flatten_features_labels(X_train, y_train)
X_test_nb_features, y_test_nb_flat = flatten_features_labels(X_test, y_test)

print("NB train tokens:", len(X_train_nb_features))
print("NB test tokens :", len(X_test_nb_features))

print("\nOriginal NB train label distribution:")
display(pd.Series(y_train_nb_flat).value_counts().to_frame("count"))


NB train tokens: 7998
NB test tokens : 1089

Original NB train label distribution:


,count
O,6579
B-HERB,233
I-HERB,223
B-TREATMENT,172
B-COMPOUND,168
B-EFFECT,123
I-EFFECT,90
B-METHOD,82
I-TREATMENT,64
B-DISEASE,61


In [18]:
# =========================================================
# 17. BALANCE TRAIN TOKENS FOR NAIVE BAYES
# =========================================================

def balance_nb_training_data(features, labels, o_ratio=2, seed=SEED):
    rng = random.Random(seed)

    entity_indices = [i for i, lab in enumerate(labels) if lab != "O"]
    o_indices = [i for i, lab in enumerate(labels) if lab == "O"]

    max_o = min(len(o_indices), len(entity_indices) * o_ratio)
    selected_o = rng.sample(o_indices, max_o)

    selected_indices = entity_indices + selected_o
    rng.shuffle(selected_indices)

    balanced_features = [features[i] for i in selected_indices]
    balanced_labels = [labels[i] for i in selected_indices]

    return balanced_features, balanced_labels


ENABLE_NB_BALANCING = True
NB_O_RATIO = 2

if ENABLE_NB_BALANCING:
    X_train_nb_balanced, y_train_nb_balanced = balance_nb_training_data(
        X_train_nb_features,
        y_train_nb_flat,
        o_ratio=NB_O_RATIO,
        seed=SEED
    )
else:
    X_train_nb_balanced, y_train_nb_balanced = X_train_nb_features, y_train_nb_flat

print("After NB balancing:")
display(pd.Series(y_train_nb_balanced).value_counts().to_frame("count"))


After NB balancing:


,count
O,2838
B-HERB,233
I-HERB,223
B-TREATMENT,172
B-COMPOUND,168
B-EFFECT,123
I-EFFECT,90
B-METHOD,82
I-TREATMENT,64
B-DISEASE,61


In [19]:
# =========================================================
# 18. TRAIN NAIVE BAYES
# =========================================================

USE_COMPLEMENT_NB = False

if USE_COMPLEMENT_NB:
    nb_classifier = ComplementNB(alpha=0.1)
else:
    nb_classifier = MultinomialNB(alpha=0.1, fit_prior=False)

nb_model = Pipeline([
    ("vectorizer", DictVectorizer(sparse=True)),
    ("classifier", nb_classifier),
])

nb_model.fit(X_train_nb_balanced, y_train_nb_balanced)

print("Naive Bayes model trained.")
print("Classifier:", nb_model.named_steps["classifier"])


Naive Bayes model trained.
Classifier: MultinomialNB(alpha=0.1, fit_prior=False)


In [20]:
# =========================================================
# 19. EVALUATE NAIVE BAYES ON CLEAN TEST SET
# =========================================================

def predict_nb_sentences(nb_model, X_sentences):
    y_pred = []

    for sent in X_sentences:
        feats = sent2features(sent)
        pred = nb_model.predict(feats).tolist()
        pred = fix_bio_sequence(pred)
        y_pred.append(pred)

    return y_pred


y_pred_nb = predict_nb_sentences(nb_model, X_test)
y_true_nb = y_test

print("FINAL CLEAN TEST - NAIVE BAYES")
print("Naive Bayes Precision:", precision_score(y_true_nb, y_pred_nb))
print("Naive Bayes Recall   :", recall_score(y_true_nb, y_pred_nb))
print("Naive Bayes F1       :", f1_score(y_true_nb, y_pred_nb))

print("\nClassification Report Naive Bayes:")
print(classification_report(y_true_nb, y_pred_nb, digits=4))


FINAL CLEAN TEST - NAIVE BAYES
Naive Bayes Precision: 0.4230769230769231
Naive Bayes Recall   : 0.868421052631579
Naive Bayes F1       : 0.5689655172413792

Classification Report Naive Bayes:
              precision    recall  f1-score   support

   BODY_PART     0.3333    0.5000    0.4000         8
    COMPOUND     0.6786    0.8837    0.7677        43
     DISEASE     0.1538    0.4444    0.2286         9
      EFFECT     0.2241    0.9286    0.3611        14
        HERB     0.8056    0.9355    0.8657        31
      METHOD     0.2037    0.9167    0.3333        12
  POPULATION     0.2903    1.0000    0.4500         9
   TREATMENT     0.6154    0.9231    0.7385        26

   micro avg     0.4231    0.8684    0.5690       152
   macro avg     0.4131    0.8165    0.5181       152
weighted avg     0.5421    0.8684    0.6408       152



In [21]:
# =========================================================
# 20. COMPARISON: BILSTM VS NAIVE BAYES
# =========================================================

comparison = pd.DataFrame({
    "Model": ["BiLSTM", "Naive Bayes"],
    "Precision": [
        precision_score(y_true_bilstm, y_pred_bilstm),
        precision_score(y_true_nb, y_pred_nb),
    ],
    "Recall": [
        recall_score(y_true_bilstm, y_pred_bilstm),
        recall_score(y_true_nb, y_pred_nb),
    ],
    "F1-score": [
        f1_score(y_true_bilstm, y_pred_bilstm),
        f1_score(y_true_nb, y_pred_nb),
    ],
})

display(comparison)


,Model,Precision,Recall,F1-score
0,BiLSTM,0.833333,0.822368,0.827815
1,Naive Bayes,0.423077,0.868421,0.568966


# Inference Helper untuk Web/Demo

Bagian ini untuk mencoba prediksi teks bebas.  
Untuk web nanti, yang dipakai adalah artifact `.keras` dan `.pkl` yang disimpan pada cell terakhir.


In [22]:
# =========================================================
# 21. INFERENCE HELPERS
# =========================================================

def clean_text_for_ner(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\.(?=\w)", ". ", text)

    text = re.sub(r"\banti\s+inflamasi\b", "antiinflamasi", text, flags=re.IGNORECASE)
    text = re.sub(r"\banti\s+bakteri\b", "antibakteri", text, flags=re.IGNORECASE)
    text = re.sub(r"\banti\s+oksidan\b", "antioksidan", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def simple_tokenize(text):
    text = clean_text_for_ner(text)
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def predict_bilstm_tokens(tokens, threshold=None, use_class_specific=True):
    if threshold is None:
        threshold = BEST_THRESHOLD

    normalized = [normalize_token(t) for t in tokens]

    x = np.zeros((1, MAX_LEN), dtype=np.int32)

    for i, tok in enumerate(normalized[:MAX_LEN]):
        x[0, i] = word2idx.get(tok, word2idx[UNK_TOKEN])

    pred_proba = bilstm_model.predict(x, verbose=0)

    if use_class_specific:
        pred_ids = apply_class_specific_threshold(pred_proba, default_threshold=threshold)[0]
    else:
        pred_ids = apply_threshold(pred_proba, threshold)[0]

    pred_tags = []

    for i in range(min(len(tokens), MAX_LEN)):
        tag = idx2tag[int(pred_ids[i])]
        if tag == PAD_TAG:
            tag = "O"
        pred_tags.append(tag)

    return fix_bio_sequence(pred_tags)


def predict_nb_tokens(tokens):
    normalized = [normalize_token(t) for t in tokens]
    pred = nb_model.predict(sent2features(normalized)).tolist()
    return fix_bio_sequence(pred)


def extract_entities(tokens, tags):
    entities = []
    current_tokens = []
    current_label = None

    for token, tag in zip(tokens, tags):
        if tag == "O":
            if current_tokens:
                entities.append({"Entity": " ".join(current_tokens), "Label": current_label})
                current_tokens = []
                current_label = None
            continue

        if "-" not in tag:
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B":
            if current_tokens:
                entities.append({"Entity": " ".join(current_tokens), "Label": current_label})
            current_tokens = [token]
            current_label = label

        elif prefix == "I":
            if current_tokens and current_label == label:
                current_tokens.append(token)
            else:
                if current_tokens:
                    entities.append({"Entity": " ".join(current_tokens), "Label": current_label})
                current_tokens = [token]
                current_label = label

    if current_tokens:
        entities.append({"Entity": " ".join(current_tokens), "Label": current_label})

    return entities


def predict_text(text, chunk_size=None):
    tokens = simple_tokenize(text)

    if chunk_size is None:
        chunk_size = MAX_LEN

    all_bilstm_tags = []

    for start in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[start:start + chunk_size]
        chunk_tags = predict_bilstm_tokens(chunk_tokens)
        all_bilstm_tags.extend(chunk_tags)

    all_bilstm_tags = all_bilstm_tags[:len(tokens)]
    all_bilstm_tags = fix_bio_sequence(all_bilstm_tags)

    nb_tags = predict_nb_tokens(tokens)
    nb_tags = nb_tags[:len(tokens)]

    min_len = min(len(tokens), len(all_bilstm_tags), len(nb_tags))

    tokens = tokens[:min_len]
    all_bilstm_tags = all_bilstm_tags[:min_len]
    nb_tags = nb_tags[:min_len]

    token_output = pd.DataFrame({
        "Token": tokens,
        "BiLSTM_Label": all_bilstm_tags,
        "NaiveBayes_Label": nb_tags,
    })

    bilstm_entities = pd.DataFrame(extract_entities(tokens, all_bilstm_tags))
    nb_entities = pd.DataFrame(extract_entities(tokens, nb_tags))

    return token_output, bilstm_entities, nb_entities


In [23]:
# =========================================================
# 22. TEST INFERENCE
# =========================================================

sample_text = "Ekstrak kunyit mengandung kurkumin dan memiliki aktivitas antiinflamasi pada lambung."

token_output, bilstm_entities, nb_entities = predict_text(sample_text)

print("Token-level output:")
display(token_output)

print("\nBiLSTM entities:")
display(bilstm_entities)

print("\nNaive Bayes entities:")
display(nb_entities)


Token-level output:


,Token,BiLSTM_Label,NaiveBayes_Label
0,Ekstrak,B-TREATMENT,B-TREATMENT
1,kunyit,B-HERB,B-HERB
2,mengandung,O,B-DISEASE
3,kurkumin,B-COMPOUND,B-COMPOUND
4,dan,O,O
5,memiliki,O,O
6,aktivitas,B-EFFECT,B-EFFECT
7,antiinflamasi,I-EFFECT,B-EFFECT
8,pada,O,O
9,lambung,B-BODY_PART,B-BODY_PART



BiLSTM entities:


,Entity,Label
0,Ekstrak,TREATMENT
1,kunyit,HERB
2,kurkumin,COMPOUND
3,aktivitas antiinflamasi,EFFECT
4,lambung,BODY_PART



Naive Bayes entities:


,Entity,Label
0,Ekstrak,TREATMENT
1,kunyit,HERB
2,mengandung,DISEASE
3,kurkumin,COMPOUND
4,aktivitas,EFFECT
5,antiinflamasi,EFFECT
6,lambung,BODY_PART


In [24]:
# =========================================================
# 23. SAVE ARTIFACTS FOR WEB
# =========================================================

ARTIFACT_DIR = "ner_bilstm_nb_split_safe_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save BiLSTM
bilstm_model.save(os.path.join(ARTIFACT_DIR, "bilstm_ner_model.keras"))

# Save Naive Bayes
with open(os.path.join(ARTIFACT_DIR, "naive_bayes_ner_model.pkl"), "wb") as f:
    pickle.dump(nb_model, f)

# Save mappings/config
mappings = {
    "word2idx": word2idx,
    "idx2tag": idx2tag,
    "tag2idx": tag2idx,
    "max_len": MAX_LEN,
    "best_threshold": BEST_THRESHOLD,
    "class_specific_thresholds": CLASS_SPECIFIC_THRESHOLDS,
    "use_class_specific_threshold": USE_CLASS_SPECIFIC_THRESHOLD,
    "lower_token": LOWER_TOKEN,
    "pad_token": PAD_TOKEN,
    "unk_token": UNK_TOKEN,
    "pad_tag": PAD_TAG,
    "valid_entity_types": sorted(list(VALID_ENTITY_TYPES)),
}

with open(os.path.join(ARTIFACT_DIR, "ner_mappings.pkl"), "wb") as f:
    pickle.dump(mappings, f)

!zip -qr ner_bilstm_nb_split_safe_artifacts.zip ner_bilstm_nb_split_safe_artifacts

print("Saved artifacts:")
!ls -lh ner_bilstm_nb_split_safe_artifacts
print("\nZIP file: ner_bilstm_nb_split_safe_artifacts.zip")


Saved artifacts:
total 7.4M
-rw-r--r-- 1 root root 5.8M Jul  2 22:00 bilstm_ner_model.keras
-rw-r--r-- 1 root root 1.6M Jul  2 22:00 naive_bayes_ner_model.pkl
-rw-r--r-- 1 root root  23K Jul  2 22:00 ner_mappings.pkl

ZIP file: ner_bilstm_nb_split_safe_artifacts.zip


In [25]:
# =========================================================
# SAVE BILSTM MODEL AS .H5
# =========================================================

bilstm_model.save("bilstm_ner_model.h5")

print("Model saved as bilstm_ner_model.h5")

Model saved as bilstm_ner_model.h5


In [26]:
# =========================================================
# SAVE BILSTM WEIGHTS ONLY
# =========================================================

bilstm_model.save_weights("bilstm_ner_weights.weights.h5")

from google.colab import files
files.download("bilstm_ner_weights.weights.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>